# Single Model — Zoom Zoom

Building on the Kaggle community's shared work. Thanks to Chris Deotte for the
Fable feature ideas and discussions, amanatar for weighted-rank research,
najiama for LightGBM, and ensemble contributors Vinay, Ravi, Lâm Huy, Kirill and
Talha, plus everyone sharing code, comments and results.

This compact feature pipeline builds on
[Mizushima Toshihiko](https://www.kaggle.com/code/mizushimatoshihiko/s6e9-competition-only-lgbm-cv-0-94610-lb-0-94636)
and its credited contributors [megayak](https://www.kaggle.com/megayak),
[najiama](https://www.kaggle.com/najiama) and [maiernator](https://www.kaggle.com/maiernator).
Zoom Zoom adds cross-fitted income statistics, local feature ablation and this
two-pass experiment. **OpenAI Codex** assisted with local experiments, feature
testing, validation and notebook implementation. Prediction-file authors and
exact versions are credited in the attached dataset's catalog.

## Run and explore

Choose `MODEL`: LightGBM, XGBoost or CatBoost. The default LightGBM uses 111
features: the compact 97 plus 15 income statistics, minus `inc_d1`.
Its original five-fold local OOF ROC AUC was **0.946131639**, a small improvement
over the 112-feature control (**0.946123500**); four folds improved, one declined.
That is exploratory local evidence, not a leaderboard score for this notebook.

Pass 1 stratifies five folds by the original labels. Pass 2 stratifies by ten
quantile bins of pass 1's OOF predictions. Those predictions choose splits only:
they never become features or labels. Encoding, fitting, early stopping and ROC
AUC always use the original labels. Pass 2 is **not independent validation**,
because its splits depend on a previous supervised run. Both fold assignments
and both sets of OOF predictions are saved; this runs ten fits of one model type.

`submission.csv` is **80% of the dataset's top scored CSV + 20% of pass 2**,
using arithmetic weights and matching IDs. The reference is selected afresh from
the catalog, with recorded display order breaking equal-score ties. Its current
public score is 0.94649; that is the reference's score, not the new output's.
`submission_history_adjusted.csv` adds the optional public scored-history
correction **after** the 80/20 blend. Both final outputs are unscored experiments.
Keep pass 1 for model evaluation; no clean OOF score exists for the external
reference, final blend or history adjustment.


In [ ]:
import hashlib, io, json, time, zipfile
import numpy as np
import pandas as pd
import lightgbm as lgb
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import TargetEncoder

MODEL = "lightgbm"                 # "lightgbm", "xgboost", "catboost"
MODEL_WEIGHT = 0.20
N_FOLDS = 5
RESTRATIFY_BINS = 10
OUTER_SEED = 42
INNER_SEED = 17
TE_SEED = 42
TE_N_SPLITS = 5
MODEL_SEED = 42
THREADS = 6
VERBOSE_EVAL = 250

# Change these input folder strings if running locally. Keep trailing slashes.
DATA_PATH = "/kaggle/input/competitions/playground-series-s6e9/"
REFERENCE_PATH = "/kaggle/input/s6e9-zoom-zoom-baseline/"
TARGET = "Will_Buy_EV"
ID = "id"
CATEGORICAL_FEATURES = [
    "Gender", "City_Type", "Current_Car_Type", "Home_Charging_Possible",
    "Subsidy_Available", "Range_Anxiety_Level",
]


In [ ]:
def build_row_local_features(frame: pd.DataFrame):
    out = frame.copy()
    income = out['Annual_Income_USD'].to_numpy(dtype=np.int64)
    commute_x10 = np.round(out['Daily_Commute_km'].to_numpy(dtype=float) * 10).astype(np.int64)
    out['inc_d1'] = (income % 10).astype('int8')
    out['inc_d2'] = (income // 10 % 10).astype('int8')
    out['inc_d3'] = (income // 100 % 10).astype('int8')
    out['inc_mod100'] = (income % 100).astype('int16')
    out['inc_mod1000'] = (income % 1000).astype('int16')
    out['km_d1'] = (commute_x10 % 10).astype('int8')
    out['km_mod100'] = (commute_x10 % 100).astype('int8')
    for divisor in (50, 100, 250, 500, 1000, 2500, 5000):
        out[f'inc_q{divisor}'] = (income // divisor).astype('int32')
    for divisor in (5, 10, 25, 50):
        out[f'km_q{divisor}'] = (commute_x10 // divisor).astype('int32')
    out['is_30k_spike'] = (income == 30000).astype('int8')
    out['is_millionaire_cliff'] = (income >= 170537).astype('int8')
    out['is_dead_zone'] = ((income >= 38000) & (income <= 42000)).astype('int8')
    out['is_env_hater'] = (out['Environmental_Concern_Level'] == 1).astype('int8')
    keys = pd.DataFrame(index=out.index)
    keys['k_inc_exact'] = income.astype(str)
    keys['k_inc100'] = (income // 100).astype(str)
    keys['k_inc1000'] = (income // 1000).astype(str)
    keys['k_km_int'] = (commute_x10 // 10).astype(str)
    for col in CATEGORICAL_FEATURES + ['Age', 'Number_of_Cars_Owned', 'Charging_Stations_Near_Home', 'Charging_Stations_Near_Work', 'Environmental_Concern_Level']:
        keys[f'k_{col}'] = out[col].astype(str).to_numpy()
    return (out.reset_index(drop=True), keys.reset_index(drop=True))


In [ ]:
def as_string(series: pd.Series) -> pd.Series:
    return series.astype(str).reset_index(drop=True)

def map_with_default_zero(values: pd.Series, mapping: pd.Series) -> np.ndarray:
    return values.map(mapping).fillna(0.0).astype(np.float32).to_numpy()

def prepare_fold_features(train_positions, valid_positions):
    train_part = X_rowlocal.iloc[train_positions].reset_index(drop=True).copy()
    valid_part = X_rowlocal.iloc[valid_positions].reset_index(drop=True).copy()
    test_part = X_test_rowlocal.reset_index(drop=True).copy()
    train_keys = X_keys.iloc[train_positions].reset_index(drop=True).copy()
    valid_keys = X_keys.iloc[valid_positions].reset_index(drop=True).copy()
    test_keys = X_test_keys.reset_index(drop=True).copy()
    income_train = pd.Series(train_part['Annual_Income_USD'].to_numpy(dtype=np.int64))
    income_valid = pd.Series(valid_part['Annual_Income_USD'].to_numpy(dtype=np.int64))
    income_test = pd.Series(test_part['Annual_Income_USD'].to_numpy(dtype=np.int64))
    income_counts = income_train.value_counts(dropna=False)
    train_part['fq_inc'] = map_with_default_zero(income_train, income_counts)
    valid_part['fq_inc'] = map_with_default_zero(income_valid, income_counts)
    test_part['fq_inc'] = map_with_default_zero(income_test, income_counts)
    commute_train = pd.Series(np.round(train_part['Daily_Commute_km'].to_numpy(dtype=float) * 10).astype(np.int64))
    commute_valid = pd.Series(np.round(valid_part['Daily_Commute_km'].to_numpy(dtype=float) * 10).astype(np.int64))
    commute_test = pd.Series(np.round(test_part['Daily_Commute_km'].to_numpy(dtype=float) * 10).astype(np.int64))
    commute_counts = commute_train.value_counts(dropna=False)
    train_part['fq_km'] = map_with_default_zero(commute_train, commute_counts)
    valid_part['fq_km'] = map_with_default_zero(commute_valid, commute_counts)
    test_part['fq_km'] = map_with_default_zero(commute_test, commute_counts)
    for key in train_keys.columns:
        frequency_map = train_keys[key].value_counts(normalize=True, dropna=False)
        feature_name = f'{key}_fe'
        train_part[feature_name] = map_with_default_zero(train_keys[key], frequency_map)
        valid_part[feature_name] = map_with_default_zero(valid_keys[key], frequency_map)
        test_part[feature_name] = map_with_default_zero(test_keys[key], frequency_map)
    for col in CATEGORICAL_FEATURES:
        train_strings = as_string(train_part[col])
        valid_strings = as_string(valid_part[col])
        test_strings = as_string(test_part[col])
        categories = sorted(train_strings.unique().tolist())
        category_set = set(categories)
        unseen_valid = set(valid_strings.unique().tolist()) - category_set
        unseen_test = set(test_strings.unique().tolist()) - category_set
        if unseen_valid or unseen_test:
            raise RuntimeError(f'Unseen category in {col}: valid={sorted(unseen_valid)}, test={sorted(unseen_test)}')
        dtype = pd.CategoricalDtype(categories=categories, ordered=False)
        train_part[col] = train_strings.astype(dtype)
        valid_part[col] = valid_strings.astype(dtype)
        test_part[col] = test_strings.astype(dtype)
    y_train = y[train_positions]
    for smooth, tag in (('auto', 'auto'), (10.0, '10'), (100.0, '100')):
        encoder = TargetEncoder(shuffle=True, cv=TE_N_SPLITS, smooth=smooth, random_state=TE_SEED)
        encoded_train = encoder.fit_transform(train_keys, y_train)
        encoded_valid = encoder.transform(valid_keys)
        encoded_test = encoder.transform(test_keys)
        for idx, key in enumerate(train_keys.columns):
            feature_name = f'{key}_te{tag}'
            train_part[feature_name] = encoded_train[:, idx].astype(np.float32)
            valid_part[feature_name] = encoded_valid[:, idx].astype(np.float32)
            test_part[feature_name] = encoded_test[:, idx].astype(np.float32)
    return (train_part, valid_part, test_part)


In [ ]:
def bin_statistics(codes, y_fit, n_bins, prior, smooth):
    sums = np.bincount(codes, weights=y_fit, minlength=n_bins).astype(np.float64)
    counts = np.bincount(codes, minlength=n_bins).astype(np.float64)
    central = (sums + smooth * prior) / (counts + smooth)
    left_sums, left_counts = (np.r_[0.0, sums[:-1]], np.r_[0.0, counts[:-1]])
    right_sums, right_counts = (np.r_[sums[1:], 0.0], np.r_[counts[1:], 0.0])
    left = (left_sums + smooth * prior) / (left_counts + smooth)
    right = (right_sums + smooth * prior) / (right_counts + smooth)
    kernel = np.exp(-0.5 * (np.arange(-1, 2) / 0.8) ** 2)
    neighbor_sums = np.convolve(sums, kernel, mode='same')
    neighbor_counts = np.convolve(counts, kernel, mode='same')
    symmetric = (neighbor_sums + smooth * kernel.sum() * prior) / (neighbor_counts + smooth * kernel.sum())
    slope = right - left
    curvature = central - 0.5 * (left + right)
    return np.column_stack([central, symmetric, left, right, slope, curvature, np.log1p(counts)]).astype(np.float32)

def income_asymmetric_features(fit_income, fit_y, valid_income, test_income, q=16384, smooth=10.0, seed=17):
    fit_income = np.asarray(fit_income, np.float64)
    valid_income = np.asarray(valid_income, np.float64)
    test_income = np.asarray(test_income, np.float64)
    fit_y = np.asarray(fit_y, np.uint8)
    prior = float(fit_y.mean())
    edges = np.linspace(float(fit_income.min()), float(fit_income.max()), q + 1)
    fit_codes = np.searchsorted(edges[1:-1], fit_income)
    valid_codes = np.searchsorted(edges[1:-1], valid_income)
    test_codes = np.searchsorted(edges[1:-1], test_income)
    n_bins = len(edges)
    position_scale = max(n_bins - 2, 1)
    fit_features = np.zeros((len(fit_income), 8), np.float32)
    inner = StratifiedKFold(5, shuffle=True, random_state=seed)
    for inner_fit, inner_valid in inner.split(fit_codes, fit_y):
        stats = bin_statistics(fit_codes[inner_fit], fit_y[inner_fit], n_bins, float(fit_y[inner_fit].mean()), smooth)
        fit_features[inner_valid, 0] = fit_codes[inner_valid] / position_scale
        fit_features[inner_valid, 1:] = stats[fit_codes[inner_valid]]
    stats = bin_statistics(fit_codes, fit_y, n_bins, prior, smooth)
    valid_features = np.column_stack([valid_codes / position_scale, stats[valid_codes]]).astype(np.float32)
    test_features = np.column_stack([test_codes / position_scale, stats[test_codes]]).astype(np.float32)
    return (fit_features, valid_features, test_features)


In [ ]:
def prepare_features(fit, valid):
    frames = list(prepare_fold_features(fit, valid))
    income = train.Annual_Income_USD.to_numpy()
    test_income = test.Annual_Income_USD.to_numpy()
    for resolution in (8192, 16384):
        blocks = income_asymmetric_features(
            income[fit], y[fit], income[valid], test_income,
            q=resolution, seed=INNER_SEED,
        )
        for frame, block in zip(frames, blocks):
            for j in range(1 if resolution == 16384 else 0, block.shape[1]):
                frame[f"own_income_{resolution}_{j}"] = block[:, j]
    return [frame.drop(columns="inc_d1") for frame in frames]


In [ ]:
def make_model(name, fold):
    if name == "lightgbm":
        return LGBMClassifier(
            n_estimators=3500, learning_rate=0.02, max_depth=5, num_leaves=32,
            min_child_samples=10, subsample=0.8, subsample_freq=1,
            colsample_bytree=0.3, reg_alpha=0.071, reg_lambda=2.0,
            max_bin=255, feature_pre_filter=False, random_state=MODEL_SEED,
            n_jobs=THREADS, verbose=-1, metric="auc",
        )
    if name == "xgboost":
        return XGBClassifier(
            n_estimators=2400, max_depth=6, learning_rate=0.03,
            min_child_weight=12, subsample=0.82, colsample_bytree=0.55,
            reg_alpha=0.08, reg_lambda=3, max_bin=512,
            objective="binary:logistic", eval_metric="auc", tree_method="hist",
            enable_categorical=True, early_stopping_rounds=120,
            random_state=MODEL_SEED + fold, n_jobs=THREADS,
        )
    if name == "catboost":
        return CatBoostClassifier(
            iterations=600, depth=6, learning_rate=0.07, l2_leaf_reg=5, rsm=0.8,
            loss_function="Logloss", eval_metric="AUC", random_seed=MODEL_SEED + fold,
            random_strength=0.35, bootstrap_type="Bayesian", bagging_temperature=0.45,
            od_type="Iter", od_wait=140, thread_count=THREADS, verbose=False,
            cat_features=CATEGORICAL_FEATURES, allow_writing_files=False,
        )
    raise ValueError("MODEL must be lightgbm, xgboost or catboost")


def fit_model(model, name, x_fit, y_fit, x_valid, y_valid):
    if name == "lightgbm":
        model.fit(x_fit, y_fit, eval_set=[(x_valid, y_valid)], eval_metric="auc",
                  callbacks=[lgb.early_stopping(100, first_metric_only=True),
                             lgb.log_evaluation(VERBOSE_EVAL)])
    elif name == "xgboost":
        model.fit(x_fit, y_fit, eval_set=[(x_valid, y_valid)], verbose=VERBOSE_EVAL)
    else:
        model.fit(x_fit, y_fit, eval_set=(x_valid, y_valid),
                  use_best_model=True, verbose=VERBOSE_EVAL)
    return model


def best_trees(model, name):
    if name == "lightgbm":
        return int(model.best_iteration_)
    if name == "xgboost":
        return int(model.best_iteration) + 1
    return int(model.get_best_iteration()) + 1


In [ ]:
train = pd.read_csv(DATA_PATH + "train.csv")
test = pd.read_csv(DATA_PATH + "test.csv")
y = (train.Will_Buy_EV.map({"No": 0, "Yes": 1})
     if not pd.api.types.is_numeric_dtype(train.Will_Buy_EV)
     else train.Will_Buy_EV).to_numpy(np.uint8)
X_rowlocal, X_keys = build_row_local_features(train.drop(columns=["id", "Will_Buy_EV"]))
X_test_rowlocal, X_test_keys = build_row_local_features(test.drop(columns="id"))

train_ids = train[ID].to_numpy()
test_ids = test[ID].to_numpy()


In [ ]:
catalog = pd.read_csv(REFERENCE_PATH + "ranked_catalog_latest.csv")
catalog["public_score"] = pd.to_numeric(catalog.public_score, errors="coerce")
catalog = catalog[np.isfinite(catalog.public_score)].sort_values(
    ["public_score", "display_order"], ascending=[False, True], kind="stable")
if catalog.empty:
    raise ValueError("No verified scored files in the dataset catalog")
best_record = catalog.iloc[0]
with zipfile.ZipFile(REFERENCE_PATH + "ranked_predictions_latest.zip.bin") as archive:
    content = archive.read(best_record.csv_path)
baseline_hash = hashlib.sha256(content).hexdigest()
if baseline_hash != best_record.sha256:
    raise ValueError("Top reference hash does not match its catalog entry")
reference = pd.read_csv(io.BytesIO(content)).set_index(ID, verify_integrity=True)
shared = reference.loc[test_ids, TARGET].to_numpy(float)
if len(reference) != len(test_ids) or not np.isfinite(shared).all():
    raise ValueError("Top reference must contain exactly the competition test IDs")
print(f"Reference: {best_record.author} | submission {int(best_record.submission_id)}"
      f" | public {best_record.public_score:.5f} | {best_record.csv_path}")


In [ ]:
def run_folds(split_labels, run_name):
    splitter = StratifiedKFold(N_FOLDS, shuffle=True, random_state=OUTER_SEED)
    fold_ids = np.full(len(train), -1, dtype=np.int8)
    oof = np.zeros(len(train), dtype=np.float64)
    test_folds, fold_rows = [], []
    started = time.time()

    for fold, (fit_idx, valid_idx) in enumerate(splitter.split(X_rowlocal, split_labels)):
        print(f"\n{run_name} | {MODEL} | fold {fold + 1}/{N_FOLDS}: preparing features", flush=True)
        x_fit, x_valid, x_test = prepare_features(fit_idx, valid_idx)
        print(f"{x_fit.shape[1]} features | {len(fit_idx):,} fit / {len(valid_idx):,} validation", flush=True)
        model = fit_model(make_model(MODEL, fold), MODEL,
                          x_fit, y[fit_idx], x_valid, y[valid_idx])
        valid_prediction = model.predict_proba(x_valid)[:, 1]
        test_prediction = model.predict_proba(x_test)[:, 1]
        fold_ids[valid_idx] = fold
        oof[valid_idx] = valid_prediction
        test_folds.append(np.asarray(test_prediction, dtype=np.float64))
        score = roc_auc_score(y[valid_idx], valid_prediction)
        fold_rows.append({"fold": fold, "ROC AUC": score, "trees": best_trees(model, MODEL),
                          "valid_id_sha256": hashlib.sha256(train_ids[valid_idx].tobytes()).hexdigest()})
        print(f"Fold ROC AUC: {score:.9f} | elapsed {(time.time() - started) / 60:.1f} min", flush=True)
        del x_fit, x_valid, x_test, model

    print(f"\n{run_name} OOF ROC AUC: {roc_auc_score(y, oof):.9f}")
    display(pd.DataFrame(fold_rows))
    return oof, test_folds, fold_rows, fold_ids


first_oof, first_test_folds, first_rows, first_fold_ids = run_folds(y, "Pass 1")

# Predictions choose outer splits only. The target y remains the original labels.
split_labels = pd.qcut(first_oof, q=RESTRATIFY_BINS, labels=False,
                       duplicates="drop").astype(np.int32)
if np.unique(split_labels).size < 2 or np.bincount(split_labels).min() < N_FOLDS:
    raise ValueError("Too few populated score bins; reduce RESTRATIFY_BINS")

oof, test_folds, fold_rows, fold_ids = run_folds(split_labels, "Pass 2")
display(pd.DataFrame({"pass": [1, 2],
                      "OOF ROC AUC": [roc_auc_score(y, first_oof), roc_auc_score(y, oof)]}))


In [ ]:
# Preserve both runs; the final model and blend use pass 2.
pd.DataFrame({ID: test_ids, TARGET: np.mean(first_test_folds, axis=0)}).to_csv(
    f"submission_{MODEL}_pass1.csv", index=False)
pd.DataFrame({ID: train_ids, TARGET: y, "oof_prediction": first_oof,
              "fold": first_fold_ids}).to_csv(f"oof_{MODEL}_pass1.csv", index=False)
pd.DataFrame(first_rows).to_csv(f"cv_scores_{MODEL}_pass1.csv", index=False)

prediction = np.mean(test_folds, axis=0)
submission = pd.DataFrame({ID: test_ids, TARGET: prediction})
submission.to_csv(f"submission_{MODEL}.csv", index=False)
pd.DataFrame({ID: train_ids, TARGET: y, "oof_prediction": oof,
              "fold": fold_ids, "split_bin": split_labels}).to_csv(f"oof_{MODEL}.csv", index=False)
pd.DataFrame(fold_rows).to_csv(f"cv_scores_{MODEL}.csv", index=False)

# 80% dataset top + 20% second-pass model, matched by id.
shared = reference.loc[test_ids, TARGET].to_numpy()
submission[TARGET] = (1 - MODEL_WEIGHT) * shared + MODEL_WEIGHT * prediction
submission.to_csv("submission.csv", index=False)
submission.head()


## Optional scored-history adjustment

This reuses the public score-slope experiment with the current catalog and archive.
It looks for author-stable record-level rank trends across nearby scored files,
then applies bounded changes **to the new 80/20 output**. It does not train on
test labels. Leaderboard scores are aggregate, rounded observations: these trends
are a heuristic, not proof that an individual prediction is more correct. Equal
displayed scores stay equal; display order does not create hidden decimals.

`submission.csv` stays the plain 80/20 result. The optional result is
`submission_history_adjusted.csv`. Change `ADJUSTMENT_STEP` to explore strength
and direction; zero reproduces the plain blend exactly. The inherited -0.25 is
a starting setting, **not a locally or publicly validated improvement for this output**.
This section can overfit public scores and has no clean OOF AUC.


In [ ]:
ADJUSTMENT_STEP = -0.25
SCORE_WINDOW = 0.00020
MAX_DIRECTION = 0.002

# Read current files directly from the archive: no cached historical extraction.
selected = catalog[catalog.public_score >= best_record.public_score - SCORE_WINDOW].copy()
selected = selected.drop_duplicates("sha256").reset_index(drop=True)
if selected.public_score.nunique() < 2:
    raise ValueError("Widen SCORE_WINDOW: at least two displayed scores are needed.")
research_ids = test_ids
current = submission[TARGET].to_numpy(float)
anchor_rank = (rankdata(current) - 0.5) / len(current)
ranks = np.empty((len(selected), len(current)), dtype=np.float32)
with zipfile.ZipFile(REFERENCE_PATH + "ranked_predictions_latest.zip.bin") as archive:
    for j, record in selected.iterrows():
        content = archive.read(record.csv_path)
        if hashlib.sha256(content).hexdigest() != record.sha256:
            raise ValueError("Archive file hash mismatch: " + record.csv_path)
        frame = pd.read_csv(io.BytesIO(content)).set_index(ID, verify_integrity=True)
        values = frame.loc[research_ids, TARGET].to_numpy(float)
        if len(frame) != len(current) or not np.isfinite(values).all():
            raise ValueError("Invalid prediction IDs or values: " + record.csv_path)
        ranks[j] = (rankdata(values) - 0.5) / len(values)

# Equal author influence, then more weight near the top displayed score.
scores = selected.public_score.to_numpy(float)
families = selected.author.to_numpy()
counts = selected.author.map(selected.author.value_counts()).to_numpy()
weights = np.exp((scores - scores.max()) / 0.0001) / counts

def score_slope(predictions, weights):
    w = weights / weights.sum()
    x = (scores - float(best_record.public_score)) / 0.0001
    dx = x - w @ x
    mean = w @ predictions
    slope = (w * dx) @ predictions / (w @ (dx * dx) + 0.01)
    residual = predictions - mean - dx[:, None] * slope
    r2 = 1 - (w @ (residual * residual)) / np.maximum(
        w @ ((predictions - mean) ** 2), 1e-12)
    return slope, np.clip(r2, 0, 1)

direction = np.zeros(len(current))
confidence = np.zeros(len(current))
for start in range(0, len(current), 4096):
    stop = min(start + 4096, len(current))
    block = ranks[:, start:stop].astype(float)
    slope, r2 = score_slope(block, weights)
    agreements = []
    for author in np.unique(families):
        other_weights = weights * (families != author)
        if other_weights.sum() and np.unique(scores[other_weights > 0]).size > 1:
            other_slope, _ = score_slope(block, other_weights)
            agreements.append(np.sign(other_slope) == np.sign(slope))
    stability = np.mean(agreements, axis=0) if agreements else np.zeros(stop - start)
    direction[start:stop] = 3.7 * slope
    confidence[start:stop] = r2 * stability * (r2 >= 0.2) * (stability >= 0.8)

# Apply bounded rank movement, then map back to the 80/20 blend's value scale.
rank_change = ADJUSTMENT_STEP * confidence * np.clip(direction, -MAX_DIRECTION, MAX_DIRECTION)
order = np.argsort(current, kind="stable")
adjusted = current.copy()
moved = rank_change != 0
adjusted[moved] = np.interp(np.clip(anchor_rank[moved] + rank_change[moved], 0, 1),
                           anchor_rank[order], current[order])
pd.DataFrame({ID: research_ids, TARGET: adjusted}).to_csv("submission_history_adjusted.csv", index=False)
changed = int(np.count_nonzero(adjusted != current))
provenance = dict(reference_file=str(best_record.csv_path),
                  reference_author=str(best_record.author),
                  reference_submission=int(best_record.submission_id),
                  reference_public_score=float(best_record.public_score),
                  reference_sha256=baseline_hash, scored_files=len(selected),
                  model=MODEL, model_weight=MODEL_WEIGHT, model_pass=2,
                  adjustment_step=ADJUSTMENT_STEP, changed_records=changed,
                  main_output="submission.csv", adjustment_output="submission_history_adjusted.csv",
                  final_output_scores="unmeasured")
with open("submission_provenance.json", "w") as file:
    json.dump(provenance, file, indent=2)
print(f"{len(selected)} scored files | {changed:,} adjusted records")
print("Saved submission.csv (80/20) and submission_history_adjusted.csv (experimental correction).")
del ranks


## Ｈ𝐀𝑷𝑷𝓎 🇰𝗮𝘨𝘨🇱𝖎Ｎɢ 💯
